# Image Captioning and Visual Q&A

Sends images to a vision-language model on watsonx.ai (Llama 3.2 Vision) to get captions and answers to questions about them.

Set `WATSONX_API_KEY` and `WATSONX_PROJECT_ID` as environment variables before running. Put your own images (file paths or URLs) in `image_sources` below.

In [ ]:
%pip install -q ibm-watsonx-ai==1.1.20 requests==2.32.0

## Model setup

In [ ]:
import os
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.foundation_models.schema import TextChatParameters

api_key = os.environ.get('WATSONX_API_KEY')
project_id = os.environ.get('WATSONX_PROJECT_ID', 'skills-network')

credentials = Credentials(
    url='https://us-south.ml.cloud.ibm.com',
    api_key=api_key
)

params = TextChatParameters(temperature=0.1, top_p=0.4)

model = ModelInference(
    model_id='meta-llama/llama-3-2-11b-vision-instruct',
    credentials=credentials,
    project_id=project_id,
    params=params,
)

## Loading images

Images are sent to the model as base64 strings inside the chat message. Each source can be a local file path or an http(s) URL.

In [ ]:
import base64
import mimetypes
import requests

def encode_image(source):
    """
    Downloads or reads an image and converts it into a Base64-encoded string.
    Automatically converts GitHub /blob/ links into raw media links.
    """
    try:
        # Automatically transform GitHub web preview URL to raw content URL
        if "github.com" in source and "/blob/" in source:
            source = source.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

        if source.startswith(('http://', 'https://')):
            response = requests.get(source, timeout=30)
            if response.status_code != 200:
                print(f"[-] Warning: Failed to fetch image (HTTP {response.status_code}) from {source}")
                return None
            data = response.content
            mime = response.headers.get('Content-Type', '').split(';')[0] or 'image/png'
        else:
            with open(source, 'rb') as f:
                data = f.read()
            mime = mimetypes.guess_type(source)[0] or 'image/png'

        return mime, base64.b64encode(data).decode('utf-8')
    except Exception as e:
        print(f"[-] Error loading image {source}: {e}")
        return None

## Asking the model about an image

In [ ]:
def analyze_cyber_screenshot(encoded_image, user_query):
    """
    Sends the base64-encoded screenshot alongside a specialized DFIR prompt
    to Llama 3.2 Vision on watsonx.ai.
    """
    if encoded_image is None:
        return "[-] Analysis aborted: Image could not be loaded."
        
    mime, data = encoded_image
    
    soc_analyst_persona = (
        "You are an expert Digital Forensics and Incident Response (DFIR) analyst and SOC Tier-3 specialist. "
        "Carefully examine the provided screenshot (e.g., event logs, network packet captures, process trees, "
        "registry keys, memory dumps, or command-line outputs).\n\n"
        "Instructions:\n"
        "1. Extract and list all observable technical indicators (IPs, ports, process names, command-line arguments, timestamps, registry paths, or hashes).\n"
        "2. Detect any anomalies, LOLBins, persistence techniques, or known adversary tactics (map to MITRE ATT&CK where applicable).\n"
        "3. Provide a clear verdict: Normal / Suspicious / Malicious.\n"
        "4. Provide recommended immediate containment or investigation next steps.\n\n"
        "Investigator's Query: "
    )
    
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': soc_analyst_persona + user_query},
                {'type': 'image_url', 'image_url': {'url': f'data:{mime};base64,{data}'}},
            ],
        }
    ]
    
    response = model.chat(messages=messages)
    return response['choices'][0]['message']['content']

## Testing Execution

In [ ]:
# GitHub Example image URL
screenshot_url = "https://github.com/Abdulah278/MyProjects/blob/b6bacf629c294a86c57f25904ec1b0d4a1bc5279/DFIR/TOCTOU%20-%20OilRig/screenshots/TOCTOU%20-%20OilRig%20(10).png"

# Load & encode image
encoded_img = encode_image(screenshot_url)

# Run the forensic investigation query
query = "What forensic artifacts are shown in this image, and is there any suspicious anomaly or adversary behavior?"
result = analyze_cyber_screenshot(encoded_img, query)

print("=== DFIR FORENSIC TRIAGE REPORT ===\n")
print(result)